# Informe analítico · Saber 11 en el Valle del Cauca
**Antes (2019), durante (2020) y después (2025) de la pandemia**

Proyecto ETL, Entrega 2. Este informe se genera **directamente desde el Data Warehouse**
(`saber11_dw_e2`), es decir, con datos que ya pasaron la validación de *Great Expectations* y la carga.
Complementa el dashboard interactivo (`dashboards/saber11_valle.html`, que actualiza el DAG) con
**pruebas estadísticas de hipótesis** que dicen si las diferencias que se ven en los gráficos son
estadísticamente significativas **y** de qué tamaño son.

Todo el texto con cifras se calcula al ejecutar el informe: si el DW cambia, el informe se actualiza solo. **Las conclusiones están en la sección 10** y las limitaciones al final.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from analisis_utils import *

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

con = conectar()
df = cargar_dw(con)
ANIOS = sorted(int(a) for a in df["anio"].dropna().unique())
ETAPA = df.drop_duplicates("anio").set_index("anio")["etapa_pandemia"].to_dict()
ESTRATOS = [f"Estrato {i}" for i in range(1, 7)]
SEMESTRE_TXT = {1: "Semestre 1", 2: "Semestre 2"}
df["semestre_txt"] = df["semestre"].map(SEMESTRE_TXT)
df["anio_etapa"] = df["anio"].map(lambda a: f"{a} ({ETAPA.get(a, '')})")
n_total = len(df)
display(Markdown(f"**{num(n_total, 0)} estudiantes** del Valle del Cauca, años {', '.join(map(str, ANIOS))}."))

## 1. Preguntas que responde el informe
Los objetivos del proyecto son comparar el desempeño en Saber 11 **antes, durante y después de la
pandemia** y entender las **brechas socioeconómicas y digitales**. Se traducen en cinco preguntas:

| | Pregunta | Prueba principal |
|---|---|---|
| **P1** | ¿Cambió el puntaje global entre 2019, 2020 y 2025? | ANOVA de Welch + comparaciones por pares |
| **P2** | ¿Qué tan grande es la brecha por estrato y cambió con el tiempo? | ANOVA de Welch, η², interacción año × estrato |
| **P3** | ¿Cuál es la brecha digital (internet y computador) y cambió? | t de Welch por año, interacción año × acceso |
| **P4** | ¿Cuál es la brecha entre colegios oficiales y no oficiales? | t de Welch por año, interacción año × naturaleza |
| **P5** | ¿En qué áreas cambió más el puntaje? | t de Welch por área, contra 2019 |

## 2. Datos
Un registro por estudiante, leído del DW uniendo la tabla de hechos con sus dimensiones
(tiempo, colegio y características socioeconómicas). **No se usan archivos intermedios**:
solo lo que ya fue validado y cargado.

In [ ]:
tabla_n = (df.groupby(["anio", "periodo", "etapa_pandemia"], observed=True)
             .agg(estudiantes=("punt_global", "size"),
                  puntaje_global_medio=("punt_global", "mean"),
                  sin_puntaje=("punt_global", lambda s: int(s.isna().sum())))
             .reset_index())
tabla_n["puntaje_global_medio"] = tabla_n["puntaje_global_medio"].round(1)
display(tabla_n)

cal = (pd.crosstab(df["semestre_txt"], df["cole_calendario"], normalize="index") * 100).round(1)
display(Markdown("**Calendario del colegio por semestre (% de estudiantes)**"))
display(cal)

In [ ]:
pB = {s: float(cal.loc[SEMESTRE_TXT[s], "B"]) if "B" in cal.columns and SEMESTRE_TXT[s] in cal.index else float("nan") for s in (1, 2)}
med_sem = df.groupby("semestre", observed=True)["punt_global"].mean()
txt = (f"**Lectura.** El semestre 1 promedia {num(med_sem.get(1, float('nan')), 1)} puntos y el semestre 2 "
       f"{num(med_sem.get(2, float('nan')), 1)}: los dos semestres son muy distintos. El calendario B representa el "
       f"{num(pB[1], 1)} % de los estudiantes del semestre 1 y el {num(pB[2], 1)} % del semestre 2")
if abs(pB[1] - pB[2]) > 10:
    txt += ", lo que sugiere que cada semestre reúne tipos de colegios distintos. "
else:
    txt += ", así que el calendario no explica por sí solo esa diferencia. "
txt += "Por eso, **para comparar años se compara siempre el mismo semestre** (P1) y no los promedios anuales a secas."
display(Markdown(txt))

## 3. Metodología estadística
* **Unidad:** el estudiante. **Variable principal:** puntaje global (0–500). Las áreas se miden de 0 a 100.
* **Dos grupos →** *t de Welch* (no supone varianzas iguales): diferencia de medias, intervalo de confianza del 95 %
  y *d de Cohen* como tamaño del efecto (|d| < 0,2 insignificante; < 0,5 pequeño; < 0,8 mediano; ≥ 0,8 grande).
* **Más de dos grupos →** *ANOVA de Welch*, con *Kruskal-Wallis* como chequeo de robustez y *η²* (proporción de la
  varianza explicada por el grupo: < 0,01 insignificante; < 0,06 pequeño; < 0,14 mediano; ≥ 0,14 grande).
* **¿La brecha cambió con el tiempo? →** prueba F de la **interacción** (modelo con y sin el término año × grupo).
* **Comparaciones múltiples:** corrección de **Bonferroni**. Nivel de significancia α = 0,05.
* **Importante:** con decenas de miles de estudiantes casi cualquier diferencia sale «significativa». Por eso la
  decisión de si una diferencia **importa** se basa en el **tamaño del efecto** y no solo en el valor p.
* **Alcance:** es un estudio observacional. Las diferencias son **asociaciones**, no causas.

## 4. P1 · ¿Cambió el puntaje global antes, durante y después de la pandemia?

In [ ]:
resumen_sem = resumen_celdas(df, "punt_global", "semestre_txt", "anio")
tabla = resumen_sem.rename(columns={"semestre_txt": "Semestre", "anio": "Año", "n": "Estudiantes",
                                    "media": "Media", "desv": "Desv. est.", "ic": "± IC 95 %"})
display(tabla.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
grafico_medias(ax, df, "punt_global", "semestre_txt", "anio", orden_x=list(SEMESTRE_TXT.values()),
               colores=COLORES_ANIO)
ax.set_title("Puntaje global medio por semestre y año (IC 95 %)")
guardar_fig(fig, "p1_puntaje_por_semestre")
plt.show()

In [ ]:
p1 = {}
for s in (1, 2):
    sub = df[df["semestre"] == s]
    p1[s] = dict(anova=comparar_grupos(sub, "punt_global", "anio"),
                 pares=pares(sub, "punt_global", "anio", orden=ANIOS))

for s in (1, 2):
    a = p1[s]["anova"]
    display(Markdown(
        f"**{SEMESTRE_TXT[s]}.** ANOVA de Welch: F({a['welch_gl1']}; {num(a['welch_gl2'], 0)}) = {num(a['welch_F'], 1)}, "
        f"p {fmt_p(a['welch_p'])}; η² = {num(a['eta2'], 4)} (efecto {a['eta2_etiqueta']}). "
        f"Kruskal-Wallis: p {fmt_p(a['kruskal_p'])}."))
    display(formatear_comparaciones(p1[s]["pares"]))

In [ ]:
inter = interaccion_f(df, "punt_global", "anio", "semestre")
med = df.groupby(["anio", "semestre"], observed=True)["punt_global"].mean().unstack()
dif_sem = (med[1] - med[2])
display(Markdown(
    "**Diferencia entre semestres dentro de cada año (semestre 1 − semestre 2):** "
    + "; ".join(f"{a}: {num(v, 1)} puntos" for a, v in dif_sem.items())
    + f". La interacción año × semestre es {'significativa' if inter['p'] < ALFA else 'no significativa'} "
    f"(F({inter['gl1']}; {num(inter['gl2'], 0)}) = {num(inter['F'], 1)}, p {fmt_p(inter['p'])})."))

In [ ]:
txt = []
for s in (1, 2):
    pr = p1[s]["pares"].set_index("comparacion")
    for comp in ("2019 vs 2020", "2019 vs 2025"):
        if comp in pr.index:
            r = pr.loc[comp]
            txt.append(f"- **{SEMESTRE_TXT[s]}, {comp}:** {num(-r['diff'], 1)} puntos "
                       f"(IC 95 %: {num(-r['ic_alto'], 1)} a {num(-r['ic_bajo'], 1)}), p ajustado {fmt_p(r['p_ajustado'])}, "
                       f"d = {num(abs(r['d']), 2)} ({r['efecto']}).")
display(Markdown("**Cambio respecto a 2019 (positivo = más puntaje):**\n\n" + "\n".join(txt)))

**Cómo leer P1.** Se compara siempre el mismo semestre entre años, para no mezclar tipos de colegios distintos.
Un valor p pequeño dice que la diferencia **no parece azar**; el *d de Cohen* dice si es **grande o pequeña** en
términos prácticos. Conviene fijarse en las dos cosas a la vez.

## 5. P2 · Brecha por estrato socioeconómico

In [ ]:
df_e = df[df["estrato"].isin(ESTRATOS)].copy()
excluidos_e = 100 * (1 - len(df_e) / len(df))

fig, ax = plt.subplots(figsize=(9, 4))
grafico_medias(ax, df_e, "punt_global", "estrato", "anio", orden_x=ESTRATOS, colores=COLORES_ANIO)
ax.set_title("Puntaje global medio por estrato y año (IC 95 %)")
guardar_fig(fig, "p2_puntaje_por_estrato")
plt.show()

filas = []
eta_estrato = {}
for anio, g in df_e.groupby("anio", observed=True):
    a = comparar_grupos(g, "punt_global", "estrato")
    eta_estrato[int(anio)] = a
    filas.append({"Año": int(anio), "Estudiantes": a["n"], "F de Welch": num(a["welch_F"], 1),
                  "p": fmt_p(a["welch_p"]), "η²": num(a["eta2"], 3), "Efecto": a["eta2_etiqueta"],
                  "p Kruskal-Wallis": fmt_p(a["kruskal_p"])})
display(pd.DataFrame(filas))

brecha_estrato = brecha_por_anio(df_e, "punt_global", "estrato", "Estrato 6", "Estrato 1")
display(Markdown("**Brecha entre el estrato 6 y el estrato 1 (puntos de puntaje global):**"))
display(formatear_comparaciones(brecha_estrato, "anio"))
inter_e = interaccion_f(df_e, "punt_global", "anio", "estrato")
display(Markdown(
    f"¿La estructura de la brecha cambió entre años? Interacción año × estrato: "
    f"F({inter_e['gl1']}; {num(inter_e['gl2'], 0)}) = {num(inter_e['F'], 1)}, p {fmt_p(inter_e['p'])}, "
    f"η² parcial = {num(inter_e['eta2_parcial'], 4)}. "
    f"(Se excluyó el {num(excluidos_e, 1)} % de estudiantes sin estrato o que no informan.)"))

## 6. P3 · Brecha digital: internet y computador en el hogar

In [ ]:
brechas_digitales = {}
inter_digital = {}
for var, nombre in (("internet", "Internet"), ("computador", "Computador")):
    d = df[df[var].isin(["Si", "No"])].copy()
    excl = 100 * (1 - len(d) / len(df))
    b = brecha_por_anio(d, "punt_global", var, "Si", "No")
    brechas_digitales[nombre] = b
    it = interaccion_f(d, "punt_global", "anio", var)
    inter_digital[nombre] = it

    display(Markdown(f"### {nombre}: hogares con acceso (Sí) frente a sin acceso (No)"))
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    grafico_medias(ax, d, "punt_global", var, "anio", orden_x=["No", "Si"], colores=COLORES_ANIO)
    ax.set_title(f"Puntaje global según acceso a {nombre.lower()} (IC 95 %)")
    guardar_fig(fig, f"p3_{nombre.lower()}")
    plt.show()

    display(formatear_comparaciones(b, "anio"))
    display(Markdown(
        f"Interacción año × {nombre.lower()}: F({it['gl1']}; {num(it['gl2'], 0)}) = {num(it['F'], 1)}, "
        f"p {fmt_p(it['p'])} ({'la brecha cambió entre años' if it['p'] < ALFA else 'sin evidencia de cambio'}; "
        f"η² parcial = {num(it['eta2_parcial'], 4)}). Excluido: {num(excl, 1)} % que no informa."))

## 7. P4 · Colegios oficiales frente a no oficiales

In [ ]:
df_n = df[df["cole_naturaleza"].isin(["OFICIAL", "NO OFICIAL"])].copy()
brecha_nat = brecha_por_anio(df_n, "punt_global", "cole_naturaleza", "NO OFICIAL", "OFICIAL")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
grafico_medias(ax, df_n, "punt_global", "cole_naturaleza", "anio", orden_x=["OFICIAL", "NO OFICIAL"],
               colores=COLORES_ANIO)
ax.set_title("Puntaje global según naturaleza del colegio (IC 95 %)")
guardar_fig(fig, "p4_naturaleza")
plt.show()

display(Markdown("**Brecha NO OFICIAL − OFICIAL (puntos de puntaje global):**"))
display(formatear_comparaciones(brecha_nat, "anio"))
it = interaccion_f(df_n, "punt_global", "anio", "cole_naturaleza")
inter_nat = it
sin_colegio = int((df["cole_naturaleza"] == "No aplica").sum())
display(Markdown(
    f"Interacción año × naturaleza: F({it['gl1']}; {num(it['gl2'], 0)}) = {num(it['F'], 1)}, p {fmt_p(it['p'])}. "
    f"Se excluyen {num(sin_colegio, 0)} estudiantes con inscripción individual (sin colegio)."))

**Una advertencia sobre P4.** La naturaleza del colegio y el estrato están muy relacionados (los colegios no oficiales
atienden, en promedio, a familias de estratos más altos). Esta brecha **no** se puede atribuir al tipo de colegio por sí solo.

## 8. P5 · ¿En qué áreas cambió más el puntaje?

In [ ]:
filas = []
for col, nombre in AREAS.items():
    base = df.loc[df["anio"] == 2019, col]
    for anio in (2020, 2025):
        r = diff_medias(df.loc[df["anio"] == anio, col], base)
        filas.append(dict(area=nombre, anio=anio, **r))
areas = pd.DataFrame(filas)
areas["p_ajustado"] = np.minimum(areas["p"] * len(areas), 1.0)
areas["efecto"] = areas["d"].map(etiqueta_d)

fig, ax = plt.subplots(figsize=(8, 4))
orden = list(AREAS.values())
for i, anio in enumerate((2020, 2025)):
    sub = areas[areas["anio"] == anio].set_index("area").loc[orden]
    pos = np.arange(len(orden)) + (i - 0.5) * 0.25
    ax.errorbar(pos, sub["diff"], yerr=[sub["diff"] - sub["ic_bajo"], sub["ic_alto"] - sub["diff"]],
                fmt="o", capsize=3, label=f"{anio} vs 2019", color=COLORES_ANIO[anio])
ax.axhline(0, color="#888", lw=0.8)
ax.set_xticks(range(len(orden)))
ax.set_xticklabels(orden, rotation=15)
ax.set_ylabel("Cambio en puntos (0–100)")
ax.set_title("Cambio del puntaje por área respecto a 2019 (IC 95 %)")
ax.legend(frameon=False)
guardar_fig(fig, "p5_cambio_por_area")
plt.show()

t = areas.copy()
t["Cambio"] = t["diff"].map(lambda v: num(v, 2))
t["IC 95 %"] = [f"[{num(a, 2)} ; {num(b, 2)}]" for a, b in zip(t["ic_bajo"], t["ic_alto"])]
t["p (Bonferroni)"] = t["p_ajustado"].map(fmt_p)
t["d de Cohen"] = t["d"].map(lambda v: num(v, 2))
display(t.rename(columns={"area": "Área", "anio": "Año vs 2019", "efecto": "Efecto"})
         [["Área", "Año vs 2019", "Cambio", "IC 95 %", "p (Bonferroni)", "d de Cohen", "Efecto"]])

## 9. Resumen de brechas por año

In [ ]:
familias = {"Estrato 6 − Estrato 1": brecha_estrato,
            "Con internet − sin": brechas_digitales["Internet"],
            "Con computador − sin": brechas_digitales["Computador"],
            "No oficial − oficial": brecha_nat}
fig, ax = plt.subplots(figsize=(9, 4.2))
grafico_brechas(ax, familias)
ax.set_title("Brechas en el puntaje global por año (diferencia de medias, IC 95 %)")
guardar_fig(fig, "resumen_brechas")
plt.show()

resumen = pd.concat({k: v.set_index("anio")["diff"].round(1) for k, v in familias.items()}, axis=1)
resumen.index.name = "Año"
display(resumen)

## 10. Conclusiones

In [ ]:
def f_brecha(tabla, anio):
    return float(tabla.loc[tabla["anio"] == anio, "diff"].iloc[0])

def cambia(tabla):
    a0, a1 = f_brecha(tabla, ANIOS[0]), f_brecha(tabla, ANIOS[-1])
    return a0, a1, a1 - a0

anual = df.groupby("anio", observed=True)["punt_global"].mean()
lineas = []

def cambio_txt(tabla, inter, nombre_brecha):
    # La interacción dice si hay cambio estadístico; el rango de la brecha dice si es grande en la práctica.
    rango = float(tabla["diff"].max() - tabla["diff"].min())
    umbral = 0.1 * float(df["punt_global"].std())      # 0,1 desviaciones estándar
    if inter["p"] < ALFA and rango >= umbral:
        return (f"La {nombre_brecha} **cambió** entre años: entre el valor más alto y el más bajo hay {num(rango, 1)} puntos "
                f"(interacción año × grupo: p {fmt_p(inter['p'])}).")
    return (f"La {nombre_brecha} se mantuvo **prácticamente igual** entre años: varió {num(rango, 1)} puntos como máximo "
            f"(interacción año × grupo: p {fmt_p(inter['p'])}; se considera relevante un cambio de al menos {num(umbral, 1)} puntos, 0,1 desviaciones estándar).")

# P1
partes = []
for s in (1, 2):
    pr = p1[s]["pares"].set_index("comparacion")
    r, r20 = pr.loc["2019 vs 2025"], pr.loc["2019 vs 2020"]
    partes.append(f"{num(-r20['diff'], 1)} puntos de 2019 a 2020 y {num(-r['diff'], 1)} de 2019 a 2025 en el {SEMESTRE_TXT[s].lower()} "
                  f"(d de 2019 a 2025 = {num(abs(r['d']), 2)}, efecto {r['efecto']})")
lineas.append("**P1 · Pandemia.** Comparando el mismo semestre, el puntaje cambió " + "; y ".join(partes) + ". "
              "Los promedios anuales ({}) no son comparables por sí solos, porque cada año mezcla proporciones distintas de estudiantes por semestre."
              .format(", ".join(f"{a}: {num(v, 1)}" for a, v in anual.items())))

# P2
e0, e1, de = cambia(brecha_estrato)
lineas.append(f"**P2 · Estrato.** La brecha entre el estrato 6 y el 1 es de {num(e0, 1)} puntos en {ANIOS[0]} y de {num(e1, 1)} en {ANIOS[-1]}. "
              f"En {ANIOS[-1]} el estrato explica el {num(100 * eta_estrato[ANIOS[-1]]['eta2'], 1)} % de la variación del puntaje "
              f"(efecto {eta_estrato[ANIOS[-1]]['eta2_etiqueta']}). " + cambio_txt(brecha_estrato, inter_e, "brecha por estrato"))

# P3
for nombre, tabla in brechas_digitales.items():
    d0, d1, dd = cambia(tabla)
    d_dur = f_brecha(tabla, 2020) if 2020 in ANIOS else float("nan")
    lineas.append(f"**P3 · {nombre}.** Los hogares con acceso puntúan {num(d0, 1)} puntos más en {ANIOS[0]}, "
                  f"{num(d_dur, 1)} en 2020 y {num(d1, 1)} en {ANIOS[-1]}. "
                  + cambio_txt(tabla, inter_digital[nombre], f"brecha de {nombre.lower()}"))

# P4
n0, n1, dn = cambia(brecha_nat)
lineas.append(f"**P4 · Oficial / no oficial.** Los colegios no oficiales superan a los oficiales por {num(n0, 1)} puntos en {ANIOS[0]} "
              f"y por {num(n1, 1)} en {ANIOS[-1]}. " + cambio_txt(brecha_nat, inter_nat, "brecha entre colegios")
              + " Es una asociación, no una causa: está mezclada con el estrato.")

# P5
mejor = areas[areas["anio"] == 2025].sort_values("diff", ascending=False).iloc[0]
peor = areas[areas["anio"] == 2020].sort_values("diff").iloc[0]
lineas.append(f"**P5 · Áreas.** El mayor aumento de 2019 a 2025 está en {mejor['area']} ({num(mejor['diff'], 2)} puntos) "
              f"y la mayor caída en 2020 está en {peor['area']} ({num(peor['diff'], 2)} puntos).")

display(Markdown("\n\n".join(lineas)))

### Limitaciones que hay que tener presentes
* **Estudio observacional:** las diferencias son asociaciones; no se puede decir qué las causó (por ejemplo, que la pandemia causó un cambio).
* **Composición distinta por semestre:** 2019, 2020 y 2025 no tienen la misma proporción de estudiantes en cada semestre, y los semestres difieren mucho entre sí; por eso las comparaciones entre años se hacen dentro del mismo semestre.
* **Datos de 2020 (API):** el segundo periodo viene codificado como `20204` y se agrupó con `20202`. Además, esa fuente **no incluye** estudiantes con inscripción individual (sin colegio), que sí están en 2019 y 2025.
* **Etiquetas no informadas:** quienes no informan estrato, internet o computador se excluyeron de la brecha correspondiente (se indica el porcentaje en cada sección).
* **Significancia y tamaño:** con muestras tan grandes, un valor p pequeño no implica una diferencia importante; por eso se reporta el tamaño del efecto.
* **Comparación del 2025:** los puntajes de distintas ediciones se asumen comparables (misma escala de 0 a 500).

## 11. Cómo reproducir este informe
1. Levantar el proyecto y correr el DAG completo (`saber11_valle_etl`), de modo que el DW tenga los tres años.
2. Generar el informe en HTML (sin el código, listo para leer):
   ```
   docker compose --profile analisis run --rm jupyter jupyter nbconvert --to html --execute --no-input informe_saber11.ipynb --output-dir /work/reports --output informe_saber11
   ```
3. O abrirlo de forma interactiva: `docker compose --profile analisis up -d jupyter` y entrar a http://localhost:8888 (token `saber11`).

El resultado queda en `reports/informe_saber11.html` y las figuras en `reports/figuras/`.